# 指数平滑：简单而有效
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：10_时间序列 | 源文件：指数平滑.py | 核心功能：简单/双重/三重指数平滑、Holt-Winters

## 概述

指数平滑用指数递减的权重对历史数据加权平均。三重指数平滑（Holt-Winters）同时建模水平、趋势和季节性。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 数学原理

### 1. 简单指数平滑（SES）

适用于无趋势、无季节性的平稳序列。预测值是过去观测的加权平均，权重指数衰减：

$$\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t$$

展开递推：

$$\hat{y}_{t+1} = \alpha \sum_{j=0}^{t-1}(1-\alpha)^j y_{t-j} + (1-\alpha)^t \hat{y}_1$$

权重 $\alpha(1-\alpha)^j$ 随 $j$ 指数衰减，$\alpha \in (0,1)$ 控制衰减速度。

- $\alpha \to 1$：几乎只看最新观测（对变化反应快，噪声大）
- $\alpha \to 0$：平均很多历史值（平滑，对变化反应慢）

### 2. 双重指数平滑（Holt 线性趋势）

在 SES 基础上增加趋势分量：

**水平方程**：

$$l_t = \alpha y_t + (1-\alpha)(l_{t-1} + b_{t-1})$$

**趋势方程**：

$$b_t = \beta(l_t - l_{t-1}) + (1-\beta)b_{t-1}$$

**预测**：

$$\hat{y}_{t+h} = l_t + h \cdot b_t$$

其中 $\alpha \in (0,1)$ 是水平平滑参数，$\beta \in (0,1)$ 是趋势平滑参数，$h$ 是预测步长。

### 3. 三重指数平滑（Holt-Winters）

在 Holt 基础上增加季节性分量，分加法和乘法两种形式。

**加法模型**（季节波动幅度恒定）：

$$l_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(l_{t-1} + b_{t-1})$$

$$b_t = \beta(l_t - l_{t-1}) + (1-\beta)b_{t-1}$$

$$s_t = \gamma(y_t - l_t) + (1-\gamma)s_{t-m}$$

$$\hat{y}_{t+h} = l_t + h \cdot b_t + s_{t-m+h}$$

**乘法模型**（季节波动幅度随趋势增长）：

$$l_t = \alpha \frac{y_t}{s_{t-m}} + (1-\alpha)(l_{t-1} + b_{t-1})$$

$$s_t = \gamma \frac{y_t}{l_t} + (1-\gamma)s_{t-m}$$

$$\hat{y}_{t+h} = (l_t + h \cdot b_t) \cdot s_{t-m+h}$$

### 4. 参数的物理意义

| 参数 | 范围 | 作用 |
|------|------|------|
| $\alpha$ | $(0,1)$ | 水平平滑：对最新观测的敏感度 |
| $\beta$ | $(0,1)$ | 趋势平滑：趋势变化的敏感度 |
| $\gamma$ | $(0,1)$ | 季节平滑：季节模式的更新速度 |
| $m$ | 正整数 | 季节周期（月=12，周=7） |

### 5. 初始值估计

指数平滑需要初始值 $l_0, b_0, s_0, \ldots, s_{-m+1}$：
- $l_0$：前几个观测的平均值
- $b_0$：前几对观测的差值平均
- $s_0$：季节性初始估计（减去趋势后的季节均值）

### 6. 模型选择

| 序列特征 | 推荐模型 |
|----------|----------|
| 无趋势无季节 | SES（简单指数平滑） |
| 有趋势无季节 | Holt（双重指数平滑） |
| 有趋势有季节（恒定幅度） | HW 加法 |
| 有趋势有季节（幅度增长） | HW 乘法 |

代码中通过对比三种模型的测试集 MSE 选择最优模型。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `SimpleExpSmoothing(train).fit()` | SES：$\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t$ |
| `ExponentialSmoothing(train, trend="add")` | Holt：$l_t, b_t$ |
| `ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12)` | HW 加法模型 |
| `ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12)` | HW 乘法模型 |
| `model.forecast(len(test))` | $\hat{y}_{T+h} = l_T + h \cdot b_T + s_{T-m+h}$ |
| `model.params["smoothing_level"]` | $\alpha$ 值 |
| `model.params["smoothing_trend"]` | $\beta$ 值 |
| `model.params["smoothing_seasonal"]` | $\gamma$ 值 |

### 1. 生成合成时间序列

运行 1. 生成合成时间序列 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 240  # 20 年月度数据

t = np.arange(n)
趋势 = 0.15 * t
季节性 = 3 * np.sin(2 * np.pi * t / 12)
噪声 = np.random.normal(0, 0.8, n)
数据 = 20 + 趋势 + 季节性 + 噪声

序列 = pd.Series(数据, index=pd.date_range("2005-01-01", periods=n, freq="MS"))
print("=" * 55)
print("指数平滑方法对比")
print("=" * 55)
print(f"数据: {n} 个月 ({序列.index[0].date()} ~ {序列.index[-1].date()})")
# --- 输出结果 ---
print(f"均值: {序列.mean():.4f}, 标准差: {序列.std():.4f}")

# 训练/测试划分
train_size = int(n * 0.8)
train, test = 序列[:train_size], 序列[train_size:]
print(f"训练集: {len(train)} 个月, 测试集: {len(test)} 个月")

### 2. 平稳性检验

检验时间序列的平稳性，决定差分阶数。

In [ ]:
result = adfuller(train.dropna(), autolag="AIC")
print(f"\n--- 训练集 ADF 检验 ---")
print(f"  ADF 统计量: {result[0]:.4f}, p 值: {result[1]:.4f} → {'平稳' if result[1] < 0.05 else '非平稳'}")

### 3. 方法一：简单指数平滑 (SES)

运行 3. 方法一：简单指数平滑 (SES) 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "-" * 55)
print("方法一: 简单指数平滑 (Simple Exponential Smoothing)")
print("-" * 55)
print("适用: 无趋势、无季节性的平稳序列")

模型_ses = SimpleExpSmoothing(train, initialization_method="estimated")
ses_fit = 模型_ses.fit(optimized=True)
print(f"  平滑参数 α: {ses_fit.params['smoothing_level']:.4f}")
print(f"  初始水平:   {ses_fit.params['initial_level']:.4f}")
print(f"  AIC:        {ses_fit.aic:.2f}")

ses_pred = ses_fit.forecast(steps=len(test))
rmse_ses = np.sqrt(mean_squared_error(test, ses_pred))
mae_ses = mean_absolute_error(test, ses_pred)
print(f"  RMSE: {rmse_ses:.4f}")
print(f"  MAE:  {mae_ses:.4f}")

### 4. 方法二：双重指数平滑 (Holt)

运行 4. 方法二：双重指数平滑 (Holt) 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "-" * 55)
print("方法二: 双重指数平滑 (Holt / Holt's Linear)")
print("-" * 55)
print("适用: 有趋势、无季节性的序列")

模型_holt = ExponentialSmoothing(train, trend="add", seasonal=None,
                                 initialization_method="estimated")
holt_fit = 模型_holt.fit(optimized=True)
print(f"  平滑参数 α: {holt_fit.params['smoothing_level']:.4f}")
print(f"  趋势参数 β: {holt_fit.params['smoothing_trend']:.4f}")
# --- 输出结果 ---
print(f"  初始水平:   {holt_fit.params['initial_level']:.4f}")
print(f"  初始趋势:   {holt_fit.params['initial_trend']:.4f}")
print(f"  AIC:        {holt_fit.aic:.2f}")

holt_pred = holt_fit.forecast(steps=len(test))
rmse_holt = np.sqrt(mean_squared_error(test, holt_pred))
mae_holt = mean_absolute_error(test, holt_pred)
print(f"  RMSE: {rmse_holt:.4f}")
print(f"  MAE:  {mae_holt:.4f}")

### 5. 方法三：Holt-Winters 加法模型

运行 5. 方法三：Holt-Winters 加法模型 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "-" * 55)
print("方法三: Holt-Winters 加法季节性模型")
print("-" * 55)
print("适用: 有趋势、有加法季节性的序列（季节幅度不随趋势变化）")

模型_hw_add = ExponentialSmoothing(train, trend="add", seasonal="add",
                                   seasonal_periods=12,
                                   initialization_method="estimated")
hw_add_fit = 模型_hw_add.fit(optimized=True)
print(f"  平滑参数 α:     {hw_add_fit.params['smoothing_level']:.4f}")
# --- 输出结果 ---
print(f"  趋势参数 β:     {hw_add_fit.params['smoothing_trend']:.4f}")
print(f"  季节参数 γ:     {hw_add_fit.params['smoothing_seasonal']:.4f}")
print(f"  初始水平:       {hw_add_fit.params['initial_level']:.4f}")
print(f"  AIC:            {hw_add_fit.aic:.2f}")

hw_add_pred = hw_add_fit.forecast(steps=len(test))
rmse_hw_add = np.sqrt(mean_squared_error(test, hw_add_pred))
mae_hw_add = mean_absolute_error(test, hw_add_pred)
print(f"  RMSE: {rmse_hw_add:.4f}")
print(f"  MAE:  {mae_hw_add:.4f}")

### 6. 方法四：Holt-Winters 乘法模型

运行 6. 方法四：Holt-Winters 乘法模型 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "-" * 55)
print("方法四: Holt-Winters 乘法季节性模型")
print("-" * 55)
print("适用: 有趋势、有乘法季节性的序列（季节幅度随趋势增长）")

模型_hw_mul = ExponentialSmoothing(train, trend="add", seasonal="mul",
                                   seasonal_periods=12,
                                   initialization_method="estimated")
hw_mul_fit = 模型_hw_mul.fit(optimized=True)
print(f"  平滑参数 α:     {hw_mul_fit.params['smoothing_level']:.4f}")
# --- 输出结果 ---
print(f"  趋势参数 β:     {hw_mul_fit.params['smoothing_trend']:.4f}")
print(f"  季节参数 γ:     {hw_mul_fit.params['smoothing_seasonal']:.4f}")
print(f"  AIC:            {hw_mul_fit.aic:.2f}")

hw_mul_pred = hw_mul_fit.forecast(steps=len(test))
rmse_hw_mul = np.sqrt(mean_squared_error(test, hw_mul_pred))
mae_hw_mul = mean_absolute_error(test, hw_mul_pred)
print(f"  RMSE: {rmse_hw_mul:.4f}")
print(f"  MAE:  {mae_hw_mul:.4f}")

### 7. 模型对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 55)
print("模型对比总结")
print("=" * 55)
print(f"{'模型':<30} {'RMSE':>8} {'MAE':>8}")
print("-" * 50)
# --- 输出结果 ---
print(f"{'简单指数平滑 (SES)':<28} {rmse_ses:>8.4f} {mae_ses:>8.4f}")
print(f"{'双重指数平滑 (Holt)':<28} {rmse_holt:>8.4f} {mae_holt:>8.4f}")
print(f"{'Holt-Winters 加法':<28} {rmse_hw_add:>8.4f} {mae_hw_add:>8.4f}")
print(f"{'Holt-Winters 乘法':<28} {rmse_hw_mul:>8.4f} {mae_hw_mul:>8.4f}")

best = min(
    [("SES", rmse_ses), ("Holt", rmse_holt), ("HW加法", rmse_hw_add), ("HW乘法", rmse_hw_mul)],
    key=lambda x: x[1],
)
print(f"\n  最优模型: {best[0]} (RMSE={best[1]:.4f})")

### 8. 最优模型预测展示

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print(f"\n--- HW 加法模型预测 vs 真实（前 10 个月） ---")
print(f"{'日期':>12}  {'真实':>8}  {'预测':>8}  {'误差':>8}")
for i in range(min(10, len(test))):
    print(f"  {str(test.index[i].date()):>10}  {test.values[i]:>8.3f}  "
          f"{hw_add_pred.values[i]:>8.3f}  {test.values[i] - hw_add_pred.values[i]:>8.3f}")

### 9. 全量模型 + 未来预测

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print(f"\n--- 全量模型拟合 & 未来 12 个月预测 ---")
全模型 = ExponentialSmoothing(序列, trend="add", seasonal="add",
                              seasonal_periods=12,
                              initialization_method="estimated")
全拟合 = 全模型.fit(optimized=True)
# --- 继续 ---
未来预测 = 全拟合.forecast(steps=12)

print(f"{'月份':>12}  {'预测值':>8}")
for i in range(12):
    print(f"  {str(未来预测.index[i].date()):>10}  {未来预测.values[i]:>8.3f}")

## 关键代码解释

```python
from statsmodels.tsa.holtwinters import ExponentialSmoothing
model = ExponentialSmoothing(ts, trend="add", seasonal="add", seasonal_periods=12)
fitted = model.fit()
```

## 注意事项

1. 参数选择影响大
2. 加法 vs 乘法季节性取决于数据特征
3. 适合中短期预测

## 总结与延伸

以上代码展示了 **指数平滑** 的完整流程。

**解读要点**：
- 观察预测曲线与实际值的 **趋势一致性**
- 关注残差是否呈现随机分布（无明显模式）
- 对比不同模型的 **MAPE / RMSE** 指标

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **ETS 模型**：指数平滑的状态空间形式
- **Theta 方法**：简单但竞赛中表现惊人
- **统计模型 vs 机器学习**：简单问题用统计模型更可解释
